# 01 — Build the analytic sample

**Role A (data).** This notebook documents how `data/processed/analytic.parquet` is built from the raw UCS *Pretrial Release Data* files and reports the checks behind each sample rule. All logic lives in `src/sample.py`; this notebook only calls it and shows aggregate counts (no single-docket records, no judge names).

Decision studied: at arraignment, a New York City judge releases the defendant (ROR), sets non-monetary conditions (NMR), sets bail, or remands. Outcome `y_detain` = bail set or remanded (1) vs. ROR / NMR (0). It is the judge's decision, not whether the person was actually held.

In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import warnings
import pandas as pd
from src import config
from src.sample import diagnostics
pd.set_option("display.width", 140)

## 1. Load

The dataset is built by `python -m src.build_dataset` (run it first): `load_raw()` reads `data/raw/NYS for Web <year>.csv.gz`, keeps NYC rows, converts column names to snake_case and checks that every year has the same columns; `build_sample()` applies the sample rules. This notebook reads the saved outputs, so it does not rebuild the data. Training 2021–22, validation 2023, test 2024; the test year appears here only in sample counts and judge-coding shares, never in outcome statistics.

In [2]:
analytic = pd.read_parquet(config.ANALYTIC_FILE)
flow = pd.read_csv(config.SAMPLE_FLOW_FILE)
print(f"analytic: {len(analytic):,} rows x {analytic.shape[1]} columns")

analytic: 502,989 rows x 149 columns


## 2. Sample flow

Rows remaining after each step, by arraignment year. `analytic.parquet` keeps every row after S3, with flags (`excl_case_type`, `nominal_bail`, `dup_cycle`) so that sensitivity analyses can relax any rule; `main_sample` marks the modeling sample.

In [3]:
flow

,step,rule,2021,2022,2023,2024,total
0,S0,All rows in the raw files,246202,286588,324619,350051,1207460
1,S1,"NYC counties (New York, Kings, Bronx, Queens, ...",124264,150585,175658,197057,647564
2,S2,Drop cases disposed at arraignment (no release...,103086,123089,141384,157741,525300
3,S3,Drop unknown release decisions (= rows in anal...,99620,117294,135082,150993,502989
4,S4,"Case type in ['Docket', 'Felony Youth Complaint']",93006,107876,124892,141002,466776
5,S5,Drop nominal bail (all first bail amounts <= $1),89213,104019,120481,135842,449555
6,S6,"One docket per arr_cycle_id (earliest year, th...",89189,104007,120457,135821,449474
7,MAIN,Main sample,89189,104007,120457,135821,449474


## 3. Outcome by year

`y_detain` in the main sample, and the same rate if nominal bail were kept and counted as 1 (sensitivity version, plan decision D5). **Outcome rates are shown for the training and validation years only; the test year (2024) is not used for any outcome statistic before the 11/15 freeze.**

In [4]:
main = analytic[analytic[config.MAIN_SAMPLE_COL]]
alt = analytic[(analytic["excl_case_type"] == 0) & (analytic["dup_cycle"] == 0)]
nt = lambda d: d[d[config.YEAR_COL] != config.TEST_YEAR]   # drop the test year for outcome statistics
main_nt, alt_nt = nt(main), nt(alt)
pd.DataFrame({
    "n_main": main_nt.groupby(config.YEAR_COL).size(),
    "y_detain_main": main_nt.groupby(config.YEAR_COL)[config.TARGET_Q1].mean().round(4),
    "n_incl_nominal": alt_nt.groupby(config.YEAR_COL).size(),
    "y_detain_incl_nominal": alt_nt.groupby(config.YEAR_COL)[config.TARGET_Q1].mean().round(4),
})

,n_main,y_detain_main,n_incl_nominal,y_detain_incl_nominal
arraign_year,,,,
2021,89189,0.1291,92982,0.1646
2022,104007,0.1295,107864,0.1606
2023,120457,0.1282,124868,0.1590


## 4. Nominal bail

The data dictionary notes that \$1 bail is often set so the defendant receives credit for time served when remanded on another case. Such a decision reflects the other case, not this one, so it is flagged and excluded from the main sample.

In [5]:
bail_set = analytic[config.RELEASE_COL].eq("Bail-set")
nt(analytic[bail_set]).groupby(config.YEAR_COL)["nominal_bail"].agg(n_bail_set="size", share_nominal="mean").round(4)

,n_bail_set,share_nominal
arraign_year,,
2021,17346,0.2363
2022,21386,0.1994
2023,24356,0.2002


## 5. Arrest cycles

`arr_cycle_id` links dockets from the same arrest. Below: which case types appear together in multi-docket cycles (after S3), and how many duplicates remain once only local-court dockets and felony youth complaints are kept.

In [6]:
cyc = analytic[analytic[config.CYCLE_COL].notna()]
size = cyc.groupby(config.CYCLE_COL).size()
combo = cyc[cyc[config.CYCLE_COL].isin(size[size > 1].index)].groupby(config.CYCLE_COL)["case_type"].agg(lambda s: " + ".join(sorted(s)))
print("Multi-docket cycles:", int((size > 1).sum()))
print(combo.value_counts().head(8).to_string())
print("\nDockets flagged dup_cycle (after S4-S5):", int(analytic["dup_cycle"].sum()))
m = main[main[config.CYCLE_COL].notna()]
span = m.groupby(config.CYCLE_COL)[config.YEAR_COL].nunique()
print("Main-sample cycles spanning more than one year (would cross train/valid/test):", int((span > 1).sum()))

Multi-docket cycles: 26472
case_type
Docket + Indictment                                        13536
Docket + Superior Criminal                                  7359
Docket + Indictment + Superior Criminal                     1674
Docket + IDV - Criminal                                     1377
Docket + Superior Court Information                         1216
Felony Youth Complaint + Indictment                          697
Docket + Indictment + Indictment                             200
Docket + Superior Court Information + Superior Criminal      102

Dockets flagged dup_cycle (after S4-S5): 81


Main-sample cycles spanning more than one year (would cross train/valid/test): 0


Most multi-docket cycles are a local-court docket plus a superior-court case for the same arrest (e.g., a felony docket that is later indicted). Restricting to local-court dockets (S4) already removes these, so S6 drops very few rows. When a cycle has dockets in two years, the earlier year's docket is kept, so a later year never changes an earlier year's sample; afterwards no arrest cycle in the main sample spans two years.

## 6. Judge codes

Judge names are replaced by `judge_id`: `J001`, `J002`, ... for judges with at least `MIN_JUDGE_CASES` main-sample cases in the training years; `other` for less frequent training-year judges; `unseen` for judges who first appear in the validation or test year. The name-to-code table is never saved.

Two versions are built: `judge_id` is coded on the training years (used for model selection on 2023); `judge_id_final` is coded on 2021–2023 (used for the final refit before the 2024 test).

In [7]:
print("MIN_JUDGE_CASES =", config.MIN_JUDGE_CASES)
print("Coded judges:", main["judge_id"].str.startswith("J").groupby(main[config.YEAR_COL]).sum().to_dict(), "(cases)")
kind = main["judge_id"].where(~main["judge_id"].str.startswith("J").fillna(False), "coded")
pd.crosstab(main[config.YEAR_COL], kind, normalize="index").round(4)

MIN_JUDGE_CASES = 100
Coded judges: {np.int64(2021): 88393, np.int64(2022): 103484, np.int64(2023): 76680, np.int64(2024): 58336} (cases)


judge_id,coded,other,unseen
arraign_year,,,
2021,0.9911,0.0089,0.0000
2022,0.9950,0.0050,0.0000
2023,0.6366,0.0582,0.3052
2024,0.4295,0.0415,0.5290


In [8]:
kind_f = main[config.JUDGE_COL_FINAL].where(~main[config.JUDGE_COL_FINAL].str.startswith("J").fillna(False), "coded")
pd.crosstab(main[config.YEAR_COL], kind_f, normalize="index").round(4)

judge_id_final,coded,other,unseen
arraign_year,,,
2021,0.9918,0.0082,0.0000
2022,0.9955,0.0045,0.0000
2023,0.9983,0.0017,0.0000
2024,0.7171,0.0011,0.2818


In [9]:
v = main[main[config.SPLIT_COL] == "valid"]
pd.crosstab(v["arraign_month"], v["judge_id"].eq("unseen"), normalize="index").round(3).rename(columns={True: "share_unseen", False: "share_seen"})

judge_id,share_seen,share_unseen
arraign_month,,
1,0.999,0.001
2,0.869,0.131
3,0.724,0.276
4,0.733,0.267
5,0.678,0.322
6,0.582,0.418
7,0.636,0.364
8,0.590,0.410
9,0.613,0.387


About 30% of 2023 main-sample cases were heard by judges who do not appear in 2021–22, and 53% of 2024 cases by judges not seen in 2021–22 (28% not seen in 2021–23). The share rises from February 2023 onward, consistent with judges newly assigned to arraignment parts. **Implication for Q2:** in out-of-time evaluation, the judge feature can only help for cases before judges seen in training; the `CASE_COURT_JUDGE` comparison should also be reported separately for cases with coded judges.

## 7. Missing data in the main sample

In [10]:
pd.DataFrame({
    "dcjs_missing": main.groupby(config.YEAR_COL)["dcjs_missing"].mean(),
    "race_unknown": main["race"].eq("Unknown").groupby(main[config.YEAR_COL]).mean(),
    "arrest_type_missing": main["arrest_type"].isna().groupby(main[config.YEAR_COL]).mean(),
    "severity_missing": main["top_severity_at_arraign"].isna().groupby(main[config.YEAR_COL]).mean(),
}).round(4)

,dcjs_missing,race_unknown,arrest_type_missing,severity_missing
arraign_year,,,,
2021,0.0093,0.0692,0.0,0.0
2022,0.0085,0.0067,0.0,0.0
2023,0.0124,0.0147,0.0,0.0
2024,0.0164,0.0051,0.0,0.0


## 8. Diagnostics summary (outcome rates exclude the test year)

In [11]:
from pprint import pprint
del alt, alt_nt, cyc, combo, m
pprint(diagnostics(analytic), sort_dicts=False, width=100)

{'rows_analytic': 502989,
 'rows_main': 449474,
 'y_detain_rate_main_by_year (train/valid only)': {np.int64(2021): 0.1291,
                                                   np.int64(2022): 0.1295,
                                                   np.int64(2023): 0.1282},
 'nominal_bail_share_of_bail_set (train/valid only)': np.float64(0.2098),
 'bail_set_without_amount': 0,
 'cycles_spanning_years': 3960,
 'rows_in_cycles_spanning_years': 8414,
 'n_coded_judges': 184,
 'judge_id_share_by_year': {(2021, 'coded'): 0.9911,
                            (2021, 'other'): 0.0089,
                            (2022, 'coded'): 0.995,
                            (2022, 'other'): 0.005,
                            (2023, 'coded'): 0.6366,
                            (2023, 'unseen'): 0.3052,
                            (2023, 'other'): 0.0582,
                            (2024, 'unseen'): 0.529,
                            (2024, 'coded'): 0.4295,
                            (2024, 'other'): 0.04